# nested-param-group-loop — faded example 3: fill the outer group loop

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `nested-param-group-loop`. Running the beacon reports progress on the `Config: nested param-group loop` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: nested param-group loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nested-param-group-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nested-param-group-loop"
DD_SUBTOPIC = "Config: nested param-group loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The defining feature of the manual step is that the OUTER loop iterates over `optimizer.param_groups` so per-group hyperparameters can be read once before touching that group's parameters. Without the outer loop you cannot honor differential learning rates.

## Faded exercise 3

Complete the outer loop of `manual_sgd_step` so it iterates over the optimizer's param groups. Fill in the outer-loop header.

**Fill in:** the outer for-loop iterating over optimizer.param_groups

In [ ]:
import torch as t

t.manual_seed(8)

a = t.nn.Parameter(t.randn(3))
b = t.nn.Parameter(t.randn(3))
opt = t.optim.SGD([
    {'params': [a], 'lr': 0.1},
    {'params': [b], 'lr': 0.4},
])
a.grad = t.ones(3)
b.grad = t.full((3,), 2.0)

def manual_sgd_step(optimizer):
    groups = optimizer.param_groups
    for group in groups:
        lr = group['lr']
        for p in group['params']:
            if p.grad is None:
                continue
            p.data.add_(p.grad, alpha=-lr)

manual_sgd_step(opt)
print(a.data, b.data)


def _test():
    t.manual_seed(80)
    a = t.nn.Parameter(t.randn(3))
    b = t.nn.Parameter(t.randn(3))
    optimizer = t.optim.SGD([
        {'params': [a], 'lr': 0.1},
        {'params': [b], 'lr': 0.4},
    ])
    ga, gb = t.randn(3), t.randn(3)
    a.grad, b.grad = ga.clone(), gb.clone()
    a0, b0 = a.data.clone(), b.data.clone()
    manual_sgd_step(optimizer)
    # both groups must be touched with their own lr
    assert t.allclose(a.data, a0 - 0.1 * ga)
    assert t.allclose(b.data, b0 - 0.4 * gb)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(8)

a = t.nn.Parameter(t.randn(3))
b = t.nn.Parameter(t.randn(3))
opt = t.optim.SGD([
    {'params': [a], 'lr': 0.1},
    {'params': [b], 'lr': 0.4},
])
a.grad = t.ones(3)
b.grad = t.full((3,), 2.0)

def manual_sgd_step(optimizer):
    groups = optimizer.param_groups
    for group in groups:
        lr = group['lr']
        for p in group['params']:
            if p.grad is None:
                continue
            p.data.add_(p.grad, alpha=-lr)

manual_sgd_step(opt)
print(a.data, b.data)
```
</details>